# Experiment E10: Reward Weight Ablation Study
**Phase 6: Experiments — Issue #34**  
**Owner/Role:** Agrima Gupta  

**Objective:** Systematically evaluate the multi-objective burst-aware reward function weights:
$$\mathcal{R}(t) = w_1 \cdot R_{\text{lat}}(t) + w_2 \cdot R_{\text{util}}(t) + w_3 \cdot R_{\text{tput}}(t) + w_4 \cdot R_{\text{sla}}(t)$$
and assess their contribution to P95 latency, P99 tail latency, server load balance, and SLA compliance under 10× flash-sale burst conditions.

### Acceptance Criteria:
1. Heatmap figure generated across $(w_{\text{lat}}, w_{\text{sla}})$ parameter grid.
2. Hypothesis H7 validated: Removing SLA penalty ($w_4=0$) causes a +20–40% increase in P99 tail latency.
3. Baseline weights $(w_1=0.4, w_2=0.2, w_3=0.2, w_4=0.2)$ confirmed optimal and locked in ADR-001 (D6).

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Ensure src is on sys.path
cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str((project_root / 'src').resolve()))

from evaluation.run_e10_ablation import (
    ABLATION_VARIANTS,
    E10_SCENARIO,
    E10_SEEDS,
    run_e10_experiments,
)

print(f'Project Root: {project_root}')
print(f'Scenario: {E10_SCENARIO}, Seeds: {E10_SEEDS}')
print(f'Ablation Variants: {list(ABLATION_VARIANTS.keys())}')

Project Root: C:\Users\Devkanti Sarkar\Desktop\DRL_Cloud_Load_Balancing_Cloud_Project_2026
Scenario: e2_10x, Seeds: (132, 133, 134)
Ablation Variants: ['baseline', 'no_rlat', 'no_rutil', 'no_rtput', 'no_rsla']


## 1. Execute / Load Experiment E10 Evaluation
Loads the 5 component ablation variants across test seeds and the sensitivity matrix.

In [2]:
output_dir = project_root / 'experiments' / 'results' / 'e10_ablation'
report_path = output_dir / 'e10_ablation_report.json'
if report_path.exists():
    with open(report_path, 'r', encoding='utf-8') as f:
        report = json.load(f)
    print(f'Loaded existing Experiment E10 results from: {report_path}')
else:
    summaries, report = run_e10_experiments(output_dir=output_dir, seeds=E10_SEEDS)
    print('Experiment E10 Completed successfully!')

Loaded existing Experiment E10 results from: C:\Users\Devkanti Sarkar\Desktop\DRL_Cloud_Load_Balancing_Cloud_Project_2026\experiments\results\e10_ablation\e10_ablation_report.json


## 2. Summary Table of Component Ablation Variants

In [3]:
df_summary = pd.DataFrame.from_dict(report['variant_means'], orient='index')
df_summary.index.name = 'variant_id'
df_summary.reset_index(inplace=True)
df_summary['variant_name'] = df_summary['variant_id'].map(lambda v: ABLATION_VARIANTS[v]['name'])
cols = ['variant_id', 'variant_name', 'p95_latency_ms', 'p99_latency_ms', 'mean_latency_ms', 'throughput_rps', 'sla_violation_rate', 'cpu_util_std', 'mean_reward']
print(df_summary[cols].to_string(index=False))

variant_id           variant_name  p95_latency_ms  p99_latency_ms  mean_latency_ms  throughput_rps  sla_violation_rate  cpu_util_std  mean_reward
  baseline Baseline (Full Reward)          100.70          105.73            86.74          598.14                 0.0        0.0008       0.6342
   no_rlat        No R_lat (w1=0)          298.75          304.56           216.57          598.14                 0.0        0.0605       0.6413
  no_rutil       No R_util (w2=0)          104.21          107.53            90.23          598.14                 0.0        0.0490       0.6674
  no_rtput       No R_tput (w3=0)          101.36          106.38            86.97          598.14                 0.0        0.0009       0.6753
   no_rsla        No R_sla (w4=0)          120.36          131.53            97.00          598.14                 0.0        0.0012       0.5209


## 3. Component Ablation Multi-Panel Visualisation
Evaluates individual contributions of $R_{\text{lat}}$, $R_{\text{util}}$, $R_{\text{tput}}$, and $R_{\text{sla}}$.

In [4]:
components_img_path = output_dir / 'ablation_components.png'
print(f'Components figure path: {components_img_path}')
assert components_img_path.exists()
img = mpimg.imread(str(components_img_path))
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(img)
ax.axis('off')
plt.show()

Components figure path: C:\Users\Devkanti Sarkar\Desktop\DRL_Cloud_Load_Balancing_Cloud_Project_2026\experiments\results\e10_ablation\ablation_components.png


C:\Users\Devkanti Sarkar\AppData\Local\Temp\ipykernel_43200\2143037878.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. 2D Sensitivity Heatmap across $(w_{\text{lat}}, w_{\text{sla}})$
Visualizes the P95 latency surface across varying latency and tail SLA weights.

In [5]:
heatmap_img_path = output_dir / 'ablation_heatmap.png'
print(f'Heatmap figure path: {heatmap_img_path}')
assert heatmap_img_path.exists()
img = mpimg.imread(str(heatmap_img_path))
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img)
ax.axis('off')
plt.show()

Heatmap figure path: C:\Users\Devkanti Sarkar\Desktop\DRL_Cloud_Load_Balancing_Cloud_Project_2026\experiments\results\e10_ablation\ablation_heatmap.png


C:\Users\Devkanti Sarkar\AppData\Local\Temp\ipykernel_43200\4039830028.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Statistical Hypothesis H7 & Optimality Verification

In [6]:
h7 = report['hypothesis_h7']
print(f'Hypothesis H7: {h7["description"]}')
print(f'  Baseline P99: {h7["baseline_p99_ms"]} ms')
print(f'  No R_sla P99: {h7["no_rsla_p99_ms"]} ms')
print(f'  P99 Increase: {h7["p99_increase_percent"]}% (Target: {h7["target_range"]})')
print(f'  H7 Confirmed: {h7["confirmed"]}')
assert h7['confirmed'], 'Hypothesis H7 failed verification'

opt = report['optimality_confirmation']
print(f'\nBaseline Optimality Confirmed: {opt["baseline_confirmed_optimal"]}')
print(f'  Baseline P95: {opt["baseline_p95_ms"]} ms')
print(f'  ADR-001 Action: {opt["adr_001_decision"]}')
assert opt['baseline_confirmed_optimal'], 'Baseline weights are not optimal'

Hypothesis H7: Removing SLA penalty from reward (w4=0) increases P99 latency by +20–40%
  Baseline P99: 105.73 ms
  No R_sla P99: 131.53 ms
  P99 Increase: 24.4% (Target: +20% to +40%)
  H7 Confirmed: True

Baseline Optimality Confirmed: True
  Baseline P95: 100.7 ms
  ADR-001 Action: Confirmed and Locked in ADR-001 (D6)
